# GeoNUSAF — TC-SegFormer on **LoveDA** (external benchmark)

**Public-benchmark transfer of the proposed model. 7 classes, `ignore_index=255`, 0.3 m/px,
512x512 random crops from 1024x1024 tiles, evaluated at full 1024.**

This notebook is the Kathmandu TC-SegFormer notebook with the *data layer* swapped and the
*model, loss, optimiser and metric code left byte-identical in intent*. That is the whole point:
the LoveDA row in the paper is a clean transfer of the method, not a retuned variant.

| What changed | Why |
|---|---|
| `NUM_CLASSES` 6 -> 7, `CLASS_NAMES` -> LoveDA | Background, Building, Road, Water, Barren, Forest, Agriculture |
| masks are **single-channel 0-7**, not BGR palette | LoveDA stores `0 = no-data`, `1..7 = classes`. Section 5 does the `-1` remap and sends `0` to 255 |
| 3-fold sequence-block CV -> **official Train/Val split** | LoveDA has a fixed public split and an official test server. Inventing folds would make the number incomparable |
| whole-tile resize -> **random 512 crop from 1024** | resizing 1024->512 halves the GSD to 0.6 m and would silently shrink roads/water below the scale the detail path was designed for |
| epoch loop -> **iteration loop** (`MAX_STEPS`) | the LoveDA convention is iterations, and it makes the compute budget explicit in the paper |
| `clDice` on (Road, River) -> **(Road, Water)** | class 2 and class 3 after the remap |
| **determinism OFF** | the sort-based backward kernel costs ~6x on this workload. Seeded and reproducible in practice, not bit-exact — say exactly that in the paper |
| added: **urban / rural split reporting** | LoveDA's whole reason for existing is the urban-rural domain gap. It is free to report and most papers skip it |
| added: **official test-server export** (section 23) | so the row can sit in the same table as the published test-set numbers |

**The control run is this same notebook.** Set `PRESET = "segformer-b0"` in section 2 and every
TC-SegFormer component switches off at once, leaving vanilla SegFormer-B0 on an identical budget.
That pair — not the absolute mIoU — is the claim.

## 1. Environment

Same stack as the Kathmandu notebook: Kaggle already has torch, cv2, scipy, skimage, sklearn, so we
add `transformers`, `huggingface_hub`, `wandb` and `fvcore`.

**Internet must be enabled.** Secrets required: `HF_TOKEN` (write scope), `WANDB_API_KEY`.

`cv2.setNumThreads(0)` matters *more* here than it did on Kathmandu. There is no `.npy` RAM cache in
this notebook — 4,191 tiles at 1024x1024x3 would be 13 GB and does not fit — so every sample is a
live PNG decode plus a live distance transform in the worker processes. cv2's internal thread pool
fighting 4 DataLoader workers on 4 vCPUs is the difference between a GPU-bound and a loader-bound run.

In [ ]:
import subprocess, sys
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("transformers>=4.40", "huggingface_hub>=0.23", "wandb", "fvcore")

import os, re, json, math, time, random, warnings, shutil, zipfile, itertools
from dataclasses import dataclass, asdict, field
from pathlib import Path
import numpy as np
import cv2
cv2.setNumThreads(0)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Config

Single `@dataclass CFG`, instantiated once as `cfg`. **`PRESET` is the only field you change between
the two runs.** `"tcsegformer"` is the proposed model; `"segformer-b0"` flips all four component
toggles off and gives you the vanilla control on a byte-identical budget.

**Budget, stated honestly.** `MAX_STEPS = 20000` at batch 16 with 512 crops is roughly 320k crop
presentations, ~2,030 passes over the 2,522 training tiles' worth of pixels. On a free T4 that is
about 3-4 h wall-clock. It is *not* the 80k-iteration schedule the published leaderboard entries use,
and the paper must say so. What it buys is a controlled A/B between two models under one budget,
which is the only comparison a 12 h session can support honestly.

**Selection vs. reporting.** Validating on all 1,669 tiles at 1024 costs ~4 min. Doing that 20 times
is over an hour of the budget spent on model selection. So checkpoint selection runs on a fixed
`VAL_SUBSET_N`-tile subset, and the number that goes in the paper comes from section 20, which
evaluates the **full** validation set once on the best checkpoint. The subset is seeded and stratified
by domain so the selection signal is not accidentally all-rural.

In [ ]:
@dataclass
class CFG:
    # ---- the only field changed between the two runs ----
    PRESET: str = "tcsegformer"        # "tcsegformer" | "segformer-b0"

    # ---- identity ----
    DATASET: str = "loveda"
    ARCH_VERSION: str = "v2-residual-loveda"   # bump on module layout change; guards HF resume
    PROTOCOL: str = "v3-slide"                 # TRAINING-RECIPE version. Bump whenever the loss
                                               # schedule, augmentation or eval protocol changes:
                                               # it is part of RUN_TAG, so a changed recipe gets a
                                               # fresh HF repo and W&B run and CANNOT silently
                                               # resume or be tabled against the old protocol.
    RESUME: bool = True
    BACKBONE: str = "nvidia/segformer-b0-finetuned-ade-512-512"
    SEED: int = 42

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/loveda"    # section 4 auto-locates Train/ and Val/ under this
    NATIVE_SIZE: int = 1024                    # LoveDA tile size on disk
    IMG_SIZE: int = 512                        # training crop
    EVAL_SIZE: int = 1024                      # validate at native resolution
    SLIDE_INFER: bool = True                   # sliding-window eval at the TRAINING crop size
    SLIDE_CROP: int = 512                      # must equal IMG_SIZE
    SLIDE_STRIDE: int = 341                    # mmseg LoveDA convention (2/3 overlap), for the reported eval
    SEL_SLIDE_STRIDE: int = 512                # non-overlapping 2x2, for the cheap selection eval
    NUM_CLASSES: int = 7
    IGNORE_INDEX: int = 255
    GSD_M: float = 0.30                        # LoveDA ground sampling distance, metres/pixel
    DOMAIN_FILTER: str = "both"                # "both" | "Urban" | "Rural"
    CROP_CAT_MAX_RATIO: float = 0.75           # reject crops dominated by one class (mmseg convention)
    CROP_TRIES: int = 10
    SCALE_JITTER: bool = True                  # crop-then-resize; the standard LoveDA regulariser
    SCALE_RANGE: tuple = (0.5, 2.0)            # mmseg ratio_range for LoveDA

    # ---- D: coarse-supervision-aware reliability weights ----
    CSA_ENABLE: bool = True
    # per-class tau: unreliable band = tau_c * local half-width. Small for the thin classes.
    # order: Background, Building, Road, Water, Barren, Forest, Agriculture
    CSA_TAU: tuple = (0.60, 0.60, 0.35, 0.35, 0.60, 0.60, 0.60)
    CSA_DMAX_M: float = 6.0                    # hard cap on the unreliable band, in GROUND METRES
    CSA_WMIN: float = 0.50                     # LoveDA annotations are precise; a 0.25 floor throws away real signal
    CSA_LOVASZ_THRESH: float = 0.30            # below this, Lovasz IGNORES the pixel (hard mask)
    CORE_THRESH: float = 0.999                 # "core" region; diagnostic only, NOT comparable to published numbers

    # ---- B: thin-structure detail path ----
    USE_DETAIL: bool = True
    DETAIL_CH: tuple = (8, 16, 32)
    FUSE_CH: int = 64
    DETAIL_GATE_INIT: float = 0.1
    DROP_PATH: float = 0.1

    # ---- C: loss ----
    EN_BETA_AUTO: bool = True
    W_CAP: float = 10.0
    LABEL_SMOOTH: float = 0.05
    LOVASZ_LAMBDA: float = 0.75
    LOVASZ_WARMUP_STEPS: int = 2000
    CLDICE_ENABLE: bool = True
    CLDICE_MU: float = 0.30
    CLDICE_START_STEP: int = 2000              # skeletonising near-empty predictions teaches nothing
    CLDICE_WARMUP_STEPS: int = 2000            # RAMP mu in; a step change shocks the optimiser and breaks the loss curve
    CLDICE_CLASSES: tuple = (2, 3)             # Road, Water
    CLDICE_ITERS: int = 5
    CLDICE_RES: int = 256

    # ---- optim ----
    BATCH_SIZE: int = 16
    EVAL_BATCH: int = 4                        # 1024 eval = 4x the pixels of a 512 train crop
    GRAD_ACCUM: int = 1
    LR_HEAD: float = 6e-5
    LR_ENCODER: float = 6e-6
    LR_NEW: float = 6e-4
    WEIGHT_DECAY: float = 1e-4
    WARMUP_STEPS: int = 500
    MAX_STEPS: int = 20000
    EVAL_EVERY: int = 1000
    PATIENCE: int = 8                          # in EVALUATIONS, not epochs
    MIN_DELTA: float = 0.002
    CLIP_NORM: float = 1.0
    EMA: bool = True
    EMA_DECAY: float = 0.999
    BALANCED_SAMPLER: bool = True
    SAMPLER_CAP: float = 8.0
    NUM_WORKERS: int = 4
    VAL_SUBSET_N: int = 400                    # selection only; section 20 uses all 1669

    # ---- session ----
    SESSION_HOURS: float = 10.5                # stop cleanly and push last.pt before Kaggle kills us

    # ---- persistence ----
    HF_USER: str = "sugam24"
    HF_PUSH: bool = True
    WANDB: bool = True
    WANDB_PROJECT: str = "geonusaf-loveda"
    OUT_DIR: str = "/kaggle/working/out"
    PUSH_LAST_EVERY: int = 2                   # in evaluations

cfg = CFG()

# ---- PRESET resolves the four ablation toggles in ONE place. A half-configured control run is
# ---- the single easiest way to produce an unpublishable comparison, so it is not left to hand-editing.
if cfg.PRESET == "segformer-b0":
    cfg.USE_DETAIL = False
    cfg.CSA_ENABLE = False
    cfg.CLDICE_ENABLE = False
    cfg.BALANCED_SAMPLER = False
elif cfg.PRESET != "tcsegformer":
    raise ValueError(f"unknown PRESET {cfg.PRESET!r}")

Path(cfg.OUT_DIR).mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Background", "Building", "Road", "Water", "Barren", "Forest", "Agriculture"]
assert len(CLASS_NAMES) == cfg.NUM_CLASSES
RUN_TAG = f"{cfg.PRESET}-loveda-{cfg.DOMAIN_FILTER.lower()}-{cfg.PROTOCOL}"
HF_REPO = f"{cfg.HF_USER}/geonusaf-{RUN_TAG}"
PX_PER_M = 1.0 / cfg.GSD_M
DMAX_PX = cfg.CSA_DMAX_M * PX_PER_M

print(json.dumps(asdict(cfg), indent=1)[:1400])
print("\nRUN_TAG:", RUN_TAG, "| HF_REPO:", HF_REPO)
print(f"toggles -> detail {cfg.USE_DETAIL} | csa {cfg.CSA_ENABLE} | "
      f"cldice {cfg.CLDICE_ENABLE} | sampler {cfg.BALANCED_SAMPLER}")
print(f"CSA band cap {cfg.CSA_DMAX_M} m at {cfg.GSD_M} m/px = {DMAX_PX:.0f} px")
assert cfg.SLIDE_CROP == cfg.IMG_SIZE, "sliding window must use the training crop size"
print(f"eval: {'sliding window ' + str(cfg.SLIDE_CROP) + '/' + str(cfg.SLIDE_STRIDE) if cfg.SLIDE_INFER else 'whole tile ' + str(cfg.EVAL_SIZE)}"
      f" | scale jitter {cfg.SCALE_RANGE if cfg.SCALE_JITTER else 'off'}")

## 3. Determinism — **off**, deliberately

`torch.use_deterministic_algorithms(True)` forces a sort-based backward kernel for the bilinear
upsamples in the decoder and the detail path, which costs roughly **6x** on this workload. At 20k
steps that is the difference between a run that fits a session and one that does not.

Everything else stays seeded: python, numpy, torch, per-sample augmentation RNG, and the validation
subset. So this is *seeded and reproducible in practice, not bit-exact*. Write that sentence in the
paper rather than claiming determinism you do not have.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True          # fixed 512 / 1024 shapes -> autotune is a free win
    torch.use_deterministic_algorithms(False)

seed_everything(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("seeded", cfg.SEED, "| device", DEVICE, "| determinism OFF (see markdown)")

## 4. LoveDA discovery and domain grouping

The canonical layout is `Train/Urban/images_png`, `Train/Rural/images_png`, and the same under `Val/`
and `Test/`. Kaggle mirrors sometimes wrap that in one more directory level or lowercase it, so this
cell **locates `images_png` by search** rather than trusting a hard-coded path, then infers the split
and domain from the two parent directory names.

Loud asserts here rather than a silent 400-image training set four hours later. The expected counts
are 2,522 train / 1,669 val / 1,796 test — if what you get is a fraction of that, the mirror is
partial and no result from it is publishable.

In [ ]:
def locate_root(root):
    root = Path(root)
    hits = [p for p in root.rglob("images_png") if p.is_dir()]
    assert hits, (f"no images_png directory found under {root}. Check that the LoveDA Kaggle "
                  f"dataset is attached and DATA_ROOT points at it.")
    return hits

IMG_DIRS = locate_root(cfg.DATA_ROOT)

def parse_dir(d):
    # d = .../<Split>/<Domain>/images_png  ->  ("Train", "Urban")
    dom = d.parent.name
    spl = d.parent.parent.name
    return spl.capitalize(), dom.capitalize()

records = []
for d in IMG_DIRS:
    spl, dom = parse_dir(d)
    if spl not in ("Train", "Val", "Test"):
        print("  skipping unrecognised split dir:", d); continue
    mdir = d.parent / "masks_png"
    for p in sorted(d.glob("*.png")):
        m = mdir / p.name
        records.append(dict(split=spl, domain=dom, stem=p.stem,
                            img=str(p), msk=str(m) if m.exists() else ""))

records = sorted(records, key=lambda r: (r["split"], r["domain"], int(re.sub(r"\D", "", r["stem"]) or 0), r["stem"]))
by_split = {}
for r in records:
    by_split.setdefault(r["split"], []).append(r)

for s in ("Train", "Val", "Test"):
    rs = by_split.get(s, [])
    dom = {}
    for r in rs: dom[r["domain"]] = dom.get(r["domain"], 0) + 1
    print(f"{s:<6} {len(rs):>5} tiles   {dom}")

EXPECT = {"Train": 2522, "Val": 1669, "Test": 1796}
for s, n in EXPECT.items():
    got = len(by_split.get(s, []))
    if got != n:
        print(f"!! {s}: found {got}, official count is {n}. A partial mirror gives an "
              f"INCOMPARABLE number -- do not publish it without explaining the discrepancy.")
assert len(by_split.get("Train", [])) > 0 and len(by_split.get("Val", [])) > 0, "need Train and Val"
assert all(r["msk"] for r in by_split["Train"] + by_split["Val"]), "some Train/Val masks are missing"
print("\nTest masks present:", sum(bool(r['msk']) for r in by_split.get('Test', [])),
      "(expected 0 -- test labels are held by the official server)")

## 5. Label encoding — the one bug that silently ruins every LoveDA reproduction

LoveDA masks are **single-channel PNGs with values 0-7**:

| raw | meaning |
|---|---|
| 0 | no-data — must become `ignore_index` |
| 1 | Background |
| 2 | Building |
| 3 | Road |
| 4 | Water |
| 5 | Barren |
| 6 | Forest |
| 7 | Agriculture |

Train on the raw values and you get an 8-class problem whose class 0 is pure no-data. Everything
still runs, the loss still falls, and the reported mIoU comes out several points low with no error
message anywhere. This is the single most common LoveDA reproduction failure, so the remap gets its
own section and its own asserts.

Two checks that actually catch it: the encoded values must be a subset of `{0..6, 255}`, and the
no-data share must be *small* — if `raw == 0` covers a large fraction of a tile, the mask is not what
this cell thinks it is.

In [ ]:
def encode_mask(raw):
    # LoveDA raw 0..7  ->  class index 0..6, with raw 0 (no-data) sent to IGNORE_INDEX
    idx = raw.astype(np.int16) - 1
    idx[raw == 0] = cfg.IGNORE_INDEX
    return idx.astype(np.uint8)

rng = np.random.RandomState(0)
n_probe = min(8, len(by_split["Train"]))
probe = [by_split["Train"][i] for i in rng.choice(len(by_split["Train"]), n_probe, replace=False)]
tot = np.zeros(cfg.NUM_CLASSES + 1, dtype=np.int64)   # last slot = no-data
for r in probe:
    raw = cv2.imread(r["msk"], cv2.IMREAD_UNCHANGED)
    img = cv2.imread(r["img"], cv2.IMREAD_COLOR)
    assert raw is not None and img is not None, r["stem"]
    assert raw.ndim == 2, (f"{r['stem']}: mask has shape {raw.shape}. LoveDA masks are single-channel "
                           f"index PNGs, not RGB. A 3-channel read means cv2 expanded it -- use IMREAD_UNCHANGED.")
    assert raw.shape == img.shape[:2], f"{r['stem']}: image {img.shape[:2]} vs mask {raw.shape}"
    assert img.shape[0] == img.shape[1] == cfg.NATIVE_SIZE, f"{r['stem']}: expected {cfg.NATIVE_SIZE}px, got {img.shape[:2]}"
    v = np.unique(raw)
    assert v.min() >= 0 and v.max() <= cfg.NUM_CLASSES, f"{r['stem']}: raw values {v} outside 0..7"
    idx = encode_mask(raw)
    assert set(np.unique(idx).tolist()).issubset(set(range(cfg.NUM_CLASSES)) | {cfg.IGNORE_INDEX})
    c = np.bincount(idx.ravel(), minlength=256)
    tot[:cfg.NUM_CLASSES] += c[:cfg.NUM_CLASSES]; tot[-1] += c[cfg.IGNORE_INDEX]

share = tot / max(tot.sum(), 1)
print(f'{"class":<13}{"share %":>9}')
for n, s in zip(CLASS_NAMES, share[:cfg.NUM_CLASSES]):
    print(f"{n:<13}{100*s:>9.2f}")
print(f'{"no-data":<13}{100*share[-1]:>9.2f}')

assert share[-1] < 0.20, (f"no-data covers {100*share[-1]:.1f}% of the probe. Either the remap is wrong "
                          f"or these are not LoveDA masks.")
assert share[:cfg.NUM_CLASSES].min() > 0 or True
print("\nOK: 0->255 remap applied, 1..7 -> 0..6, all encoded values legal.")

## 6. One-time statistics scan — per-tile class counts and presence

The balanced sampler needs a per-tile class-presence table and the effective-number class weights
need per-class pixel counts, so both come from one pass over the Train and Val masks. Mask PNGs are
small (index images, not RGB) so 4,191 decodes is about 90 s, and the result is cached to
`stats.npz` and pushed to the Hub so a resumed session skips it entirely.

Note what is **not** cached: the images. 4,191 tiles at 1024x1024x3 is 13 GB, which fits neither the
~13 GB of RAM nor a sane `/kaggle/working` footprint. Images are decoded live in the DataLoader
workers — see section 7 for why that is affordable.

In [ ]:
STATS_P = Path(cfg.OUT_DIR) / "stats.npz"

def scan_masks(rows, desc):
    n = len(rows)
    counts = np.zeros((n, cfg.NUM_CLASSES), dtype=np.int64)
    for i, r in enumerate(tqdm(rows, desc=desc)):
        raw = cv2.imread(r["msk"], cv2.IMREAD_UNCHANGED)
        idx = encode_mask(raw)
        counts[i] = np.bincount(idx.ravel(), minlength=256)[:cfg.NUM_CLASSES]
    return counts

if STATS_P.exists():
    z = np.load(STATS_P, allow_pickle=True)
    if list(z["train_stems"]) == [r["stem"] for r in by_split["Train"]]:
        TR_COUNTS, VA_COUNTS = z["tr"], z["va"]
        print("stats cache hit ->", STATS_P)
    else:
        STATS_P.unlink(); print("stats cache was built from a different file ordering; rebuilding")

if not STATS_P.exists():
    t0 = time.time()
    TR_COUNTS = scan_masks(by_split["Train"], "scan train")
    VA_COUNTS = scan_masks(by_split["Val"],   "scan val")
    np.savez_compressed(STATS_P, tr=TR_COUNTS, va=VA_COUNTS,
                        train_stems=np.array([r["stem"] for r in by_split["Train"]]),
                        val_stems=np.array([r["stem"] for r in by_split["Val"]]))
    print(f"scan done in {(time.time()-t0)/60:.1f} min ->", STATS_P)

TR_PRESENCE = TR_COUNTS > 0
tr_share = TR_COUNTS.sum(0) / TR_COUNTS.sum()
tile_presence_freq = TR_PRESENCE.mean(0)
print(f'\n{"class":<13}{"pixel %":>9}{"tiles w/ class %":>19}')
for i, n in enumerate(CLASS_NAMES):
    print(f"{n:<13}{100*tr_share[i]:>9.2f}{100*tile_presence_freq[i]:>19.1f}")
print("\nBarren is the diagnostic class for this dataset -- published IoU for it ranges from ~10 to ~28.")

## 7. CSA reliability weights — computed on the fly

On Kathmandu the CSA weight map was precomputed into a memmapped array because the tiles were
2345x2351 and the transform was expensive. Here it runs **inside the DataLoader worker**, on the full
1024 mask, before cropping.

Two reasons that is the right call and not a compromise:

* **It fits the budget.** A distance transform plus per-class connected components on 1024x1024 is
  ~25-30 ms; add ~12 ms to decode the image PNG and a sample costs ~45 ms. Sixteen samples across 4
  workers is ~180 ms, against a ~300-400 ms GPU step for B0 at 512. The loader stays behind the GPU.
* **Computing it before the crop is geometrically correct.** The band width for a component is a
  function of that component's local half-width. Compute it on a 512 crop and every component clipped
  by the crop edge gets a wrong, systematically-too-small width.

The `dmax` cap is expressed in **ground metres** and converted with LoveDA's 0.3 m/px, exactly as the
Kathmandu version converted with its own tile geometry. Same 6 m band, different pixel count.

In [ ]:
from scipy import ndimage as ndi

def csa_weight(idx, taus, wmin, dmax_px, n_classes):
    lab = idx.astype(np.int16)
    b = np.zeros(lab.shape, dtype=bool)
    d = lab[:-1, :] != lab[1:, :]; b[:-1, :] |= d; b[1:, :] |= d
    d = lab[:, :-1] != lab[:, 1:]; b[:, :-1] |= d; b[:, 1:] |= d
    D = cv2.distanceTransform((~b).astype(np.uint8), cv2.DIST_L2, 3)
    W = np.ones(lab.shape, dtype=np.float32)
    for c in range(n_classes):
        m = (idx == c).astype(np.uint8)
        if m.sum() == 0:
            continue
        ncc, cc = cv2.connectedComponents(m, connectivity=8)
        if ncc <= 1:
            continue
        means = np.asarray(ndi.mean(D, labels=cc, index=np.arange(1, ncc)), dtype=np.float32)
        R = 2.0 * means                                        # local half-width of the component
        band = np.maximum(np.minimum(taus[c] * R, dmax_px), 1.0)
        bmap = np.zeros(ncc, dtype=np.float32); bmap[1:] = band
        sel = m.astype(bool)
        W[sel] = np.clip(D[sel] / bmap[cc[sel]], wmin, 1.0)
    W[idx == cfg.IGNORE_INDEX] = 1.0
    return W

# timing probe -- if this is much over ~40 ms the loader will become the bottleneck and
# NUM_WORKERS should go up (or CSA_ENABLE down, which changes the model, so prefer workers).
_r = by_split["Train"][0]
_raw = cv2.imread(_r["msk"], cv2.IMREAD_UNCHANGED); _idx = encode_mask(_raw)
t0 = time.time()
for _ in range(5):
    _ = cv2.imread(_r["img"], cv2.IMREAD_COLOR)
    _ = csa_weight(_idx, cfg.CSA_TAU, cfg.CSA_WMIN, DMAX_PX, cfg.NUM_CLASSES)
per = (time.time() - t0) / 5 * 1000
print(f"decode + CSA per sample: {per:.0f} ms | est. loader throughput with {cfg.NUM_WORKERS} workers: "
      f"{1000*cfg.NUM_WORKERS/max(per,1e-6):.0f} samples/s "
      f"({cfg.BATCH_SIZE*per/cfg.NUM_WORKERS:.0f} ms per batch of {cfg.BATCH_SIZE})")

## 8. Split — the official one, and no cross-validation

**This is the point of using LoveDA, so it is worth being explicit.** The Kathmandu notebook invents
3-fold sequence-block CV because that dataset has no public split and export order is the only spatial
proxy available. LoveDA already has a fixed public Train/Val/Test partition and an official test
server. Re-splitting it would produce a number that cannot sit in a table next to any published
result, which would defeat the entire reason for running this benchmark.

So: train on `Train`, select on a seeded subset of `Val`, report on all of `Val`, and optionally
submit `Test` predictions to the server (section 23).

The selection subset is **stratified by domain** — a uniform 400-of-1669 draw would be ~59% rural by
construction, and selecting a checkpoint on a mostly-rural signal is a quiet way to lose urban
building IoU.

In [ ]:
TRAIN_ROWS = [r for r in by_split["Train"]
              if cfg.DOMAIN_FILTER == "both" or r["domain"] == cfg.DOMAIN_FILTER]
VAL_ROWS   = [r for r in by_split["Val"]
              if cfg.DOMAIN_FILTER == "both" or r["domain"] == cfg.DOMAIN_FILTER]
TRAIN_IDX  = np.array([i for i, r in enumerate(by_split["Train"])
                       if cfg.DOMAIN_FILTER == "both" or r["domain"] == cfg.DOMAIN_FILTER])

# domain-stratified selection subset
_r = np.random.RandomState(cfg.SEED)
sel = []
for dom in sorted({r["domain"] for r in VAL_ROWS}):
    pool = [i for i, r in enumerate(VAL_ROWS) if r["domain"] == dom]
    k = max(1, round(cfg.VAL_SUBSET_N * len(pool) / len(VAL_ROWS)))
    sel += _r.choice(pool, size=min(k, len(pool)), replace=False).tolist()
SEL_IDX = sorted(sel)
SEL_ROWS = [VAL_ROWS[i] for i in SEL_IDX]

def domcount(rows):
    d = {}
    for r in rows: d[r["domain"]] = d.get(r["domain"], 0) + 1
    return d

print("train        ", len(TRAIN_ROWS), domcount(TRAIN_ROWS))
print("val (report) ", len(VAL_ROWS),   domcount(VAL_ROWS))
print("val (select) ", len(SEL_ROWS),   domcount(SEL_ROWS))
print("\nOfficial public split -- no folds, no re-partitioning. Report it as such.")

## 9. Class weights — effective number, with the beta saturation trap handled loudly

Unchanged from the Kathmandu notebook, and it matters just as much here. Cui et al. apply
`E_c = (1 - beta^n) / (1 - beta)` to per-class **image** counts, where `beta = 0.9999` works because
`1/(1-beta) = 1e4` matches the count scale. Applied to **pixel** counts (here 2.6e9 total),
`beta^n` underflows to 0 for every class, `E_c` collapses to a constant, and the weights come out
exactly uniform — silently unweighted CE.

`EN_BETA_AUTO=True` sets `beta = 1 - 1/N_total`, which recovers a real spread before capping.
Barren and Water should come out with the largest weights; Background and Agriculture the smallest.

In [ ]:
def effective_number_weights(counts_per_class, beta_auto, cap):
    n = counts_per_class.astype(np.float64)
    N = n.sum()
    beta = 1.0 - 1.0 / N if beta_auto else 0.9999
    with np.errstate(over="ignore"):
        eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    ratio = eff.max() / max(eff.min(), 1e-12)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    w = np.minimum(w, cap)
    w = w / w.mean()
    return w.astype(np.float32), beta, ratio

train_counts = TR_COUNTS[TRAIN_IDX].sum(0)
CLASS_W, BETA, EFF_RATIO = effective_number_weights(train_counts, cfg.EN_BETA_AUTO, cfg.W_CAP)
if EFF_RATIO < 1.01:
    print("!! effective-number SATURATION: weights are effectively uniform. "
          "Set EN_BETA_AUTO=True or the CE term is silently unweighted.")
print(f"beta {BETA:.12f} | eff spread {EFF_RATIO:.2f}x | cap {cfg.W_CAP}")
print(f'\n{"class":<13}{"pixels":>14}{"weight":>9}')
for i, n in enumerate(CLASS_NAMES):
    print(f"{n:<13}{train_counts[i]:>14,}{CLASS_W[i]:>9.3f}")
CLASS_W_T = torch.tensor(CLASS_W, device=DEVICE)

## 10. Dataset, augmentation, and the class-presence-balanced sampler

**Random 512 crop from the 1024 tile, not a resize.** Resizing 1024 -> 512 doubles the GSD to 0.6 m,
which halves the pixel width of every road and stream. The detail path exists precisely to recover
structures that are a handful of pixels wide at H/4; feeding it pre-shrunk structures would test a
different problem than the one the module was designed for. `CROP_CAT_MAX_RATIO = 0.75` rejects crops
where a single class covers more than 75% of valid pixels (up to 10 tries), the mmseg convention —
without it, a large share of rural crops are pure agriculture and Barren is starved further.

**Augmentation is identical to Kathmandu minus the baked-in flips.** LoveDA has no pre-flipped copies
on disk, so hflip and vflip are applied on the fly here alongside `rot90` — together they are the same
D4 dihedral group the Kathmandu pipeline realised through baked flips plus rot90. Then +/-10%
brightness and contrast. **No scale jitter** (`SCALE_JITTER=False`) for the same GSD reason.

Augmentation is seeded per `(step, index)` so a resumed session reproduces the same stream.

Validation loads the **full 1024 tile, unaugmented**. SegFormer has no learned position embeddings
(Mix-FFN instead) and the detail path is fully convolutional, so evaluating at 2x the training
resolution is sound — and it is what the published numbers do.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def _rand_crop(img, idx, wgt, S, r, cat_max_ratio, tries):
    H, W = idx.shape
    if H == S and W == S:
        return img, idx, wgt
    y = x = 0
    for _ in range(max(1, tries)):
        y = int(r.randint(0, H - S + 1)); x = int(r.randint(0, W - S + 1))
        if cat_max_ratio >= 1.0:
            break
        c = idx[y:y + S, x:x + S]
        v = c[c != cfg.IGNORE_INDEX]
        if v.size == 0:
            continue
        if np.bincount(v, minlength=cfg.NUM_CLASSES).max() / v.size < cat_max_ratio:
            break
    return img[y:y + S, x:x + S], idx[y:y + S, x:x + S], wgt[y:y + S, x:x + S]

class LoveDADataset(Dataset):
    def __init__(self, rows, train, cfg):
        self.rows, self.train, self.cfg, self.epoch = rows, train, cfg, 0
    def set_epoch(self, e): self.epoch = e
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        rec = self.rows[i]
        img = cv2.imread(rec["img"], cv2.IMREAD_COLOR)[:, :, ::-1].astype(np.float32) / 255.0  # BGR->RGB
        raw = cv2.imread(rec["msk"], cv2.IMREAD_UNCHANGED)
        idx = encode_mask(raw)
        if self.cfg.CSA_ENABLE:
            wgt = csa_weight(idx, self.cfg.CSA_TAU, self.cfg.CSA_WMIN, DMAX_PX, self.cfg.NUM_CLASSES)
        else:
            wgt = np.ones(idx.shape, dtype=np.float32)

        if self.train:
            r = np.random.RandomState((self.epoch * 100003 + i) % (2 ** 31 - 1))
            S = self.cfg.IMG_SIZE
            # scale jitter as crop-then-resize: sample a window of side S/s from the native
            # tile and rescale it to S. Equivalent to resize-then-crop but a fraction of the cost,
            # and CSA has already been computed at native scale so its geometry stays correct.
            cs = S
            if self.cfg.SCALE_JITTER:
                sc = float(r.uniform(*self.cfg.SCALE_RANGE))
                cs = int(np.clip(round(S / sc), 64, idx.shape[0]))
            img, idx, wgt = _rand_crop(img, idx, wgt, cs, r,
                                       self.cfg.CROP_CAT_MAX_RATIO, self.cfg.CROP_TRIES)
            if cs != S:
                ip = cv2.INTER_LINEAR if cs < S else cv2.INTER_AREA
                img = cv2.resize(img, (S, S), interpolation=ip)
                idx = cv2.resize(idx, (S, S), interpolation=cv2.INTER_NEAREST)
                wgt = cv2.resize(wgt, (S, S), interpolation=ip)
            if r.rand() < 0.5:
                img = img[:, ::-1]; idx = idx[:, ::-1]; wgt = wgt[:, ::-1]
            if r.rand() < 0.5:
                img = img[::-1, :]; idx = idx[::-1, :]; wgt = wgt[::-1, :]
            k = int(r.randint(4))
            if k:
                img = np.rot90(img, k, (0, 1)); idx = np.rot90(idx, k); wgt = np.rot90(wgt, k)
            b = r.uniform(-0.10, 0.10); c = 1.0 + r.uniform(-0.10, 0.10)
            img = np.clip((img - 0.5) * c + 0.5 + b, 0.0, 1.0)
        elif self.cfg.EVAL_SIZE != img.shape[0]:
            S = self.cfg.EVAL_SIZE
            img = cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA)
            idx = cv2.resize(idx, (S, S), interpolation=cv2.INTER_NEAREST)
            wgt = cv2.resize(wgt, (S, S), interpolation=cv2.INTER_AREA)

        img = (img - IMNET_MEAN) / IMNET_STD
        return (torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(idx.astype(np.int64))),
                torch.from_numpy(np.ascontiguousarray(wgt.astype(np.float32))))

def make_sampler(presence, tile_freq, cap):
    inv = 1.0 / np.maximum(tile_freq, 1e-6)
    w = np.array([max(inv[c] for c in range(cfg.NUM_CLASSES) if presence[j, c]) if presence[j].any() else 1.0
                  for j in range(presence.shape[0])], dtype=np.float64)
    w = np.minimum(w / w.min(), cap)
    return WeightedRandomSampler(torch.as_tensor(w), num_samples=len(w), replacement=True)

train_ds = LoveDADataset(TRAIN_ROWS, True, cfg)
sel_ds   = LoveDADataset(SEL_ROWS,  False, cfg)
val_ds   = LoveDADataset(VAL_ROWS,  False, cfg)
sampler  = make_sampler(TR_PRESENCE[TRAIN_IDX], tile_presence_freq, cfg.SAMPLER_CAP) if cfg.BALANCED_SAMPLER else None

train_dl = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler, shuffle=(sampler is None),
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
                      persistent_workers=cfg.NUM_WORKERS > 0, prefetch_factor=4 if cfg.NUM_WORKERS > 0 else None)
sel_dl   = DataLoader(sel_ds, batch_size=cfg.EVAL_BATCH, shuffle=False, num_workers=cfg.NUM_WORKERS,
                      pin_memory=True, persistent_workers=cfg.NUM_WORKERS > 0)
val_dl   = DataLoader(val_ds, batch_size=cfg.EVAL_BATCH, shuffle=False, num_workers=cfg.NUM_WORKERS,
                      pin_memory=True)

xb, yb, wb = next(iter(train_dl))
print("train batch:", tuple(xb.shape), "|", tuple(yb.shape),
      "labels", sorted(set(yb.unique().tolist()))[:9], "| w range",
      round(float(wb.min()), 3), round(float(wb.max()), 3))
assert set(yb.unique().tolist()).issubset(set(range(cfg.NUM_CLASSES)) | {cfg.IGNORE_INDEX})
assert xb.shape[-1] == cfg.IMG_SIZE

## 11. Loss — weighted CE (x CSA) + Lovasz (CSA-masked, warm-ramped) + soft-clDice

Byte-identical to the Kathmandu notebook except that the ramps are keyed on **global step** instead of
epoch (there are no epochs here) and `CLDICE_CLASSES` is `(2, 3)` = Road, Water.

The three design decisions carry over unchanged:

**1. The whole loss is fp32 outside autocast.** Lovasz sorts per-pixel errors and its gradient is a
cumsum difference, which accumulates visible error and can NaN in fp16. `ignore_index` is filtered
before flattening so 255 never reaches either term.

**2. CSA enters CE as a weight but Lovasz as a hard mask.** Lovasz is a set-function surrogate over
the *sorted* error vector — there is no principled way to scale individual pixels inside it.

**3. soft-clDice starts late** (`CLDICE_START_STEP = 4000`). Skeletonising a near-empty Road
prediction teaches nothing and the gradient is noise.

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jacc = 1.0 - inter / union
    if p > 1:
        jacc[1:p] = jacc[1:p] - jacc[0:-1]
    return jacc

def lovasz_softmax_flat(probs, labels, classes="present"):
    if probs.numel() == 0:
        return probs * 0.0
    C = probs.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if classes == "present" and fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg[perm])))
    if not losses:
        return probs.sum() * 0.0
    return torch.stack(losses).mean()

def soft_erode(x):
    p1 = -F.max_pool2d(-x, (3, 1), (1, 1), (1, 0))
    p2 = -F.max_pool2d(-x, (1, 3), (1, 1), (0, 1))
    return torch.min(p1, p2)

def soft_open(x):
    return F.max_pool2d(soft_erode(x), (3, 3), (1, 1), (1, 1))

def soft_skel(x, iters):
    x1 = soft_open(x); skel = F.relu(x - x1)
    for _ in range(iters):
        x = soft_erode(x); x1 = soft_open(x)
        delta = F.relu(x - x1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

def soft_cldice(prob_c, gt_c, iters, eps=1.0):
    sp, sg = soft_skel(prob_c, iters), soft_skel(gt_c, iters)
    tprec = (torch.sum(sp * gt_c, (1, 2, 3)) + eps) / (torch.sum(sp, (1, 2, 3)) + eps)
    tsens = (torch.sum(sg * prob_c, (1, 2, 3)) + eps) / (torch.sum(sg, (1, 2, 3)) + eps)
    return 1.0 - 2.0 * tprec * tsens / (tprec + tsens)

class GeoLoss(nn.Module):
    def __init__(self, cfg, class_w):
        super().__init__(); self.cfg = cfg; self.register_buffer("cw", class_w); self.lam = 0.0; self.mu = 0.0
    def set_step(self, s):
        c = self.cfg
        self.lam = c.LOVASZ_LAMBDA * min(1.0, (s + 1) / max(c.LOVASZ_WARMUP_STEPS, 1))
        if c.CLDICE_ENABLE and s >= c.CLDICE_START_STEP:
            r = min(1.0, (s - c.CLDICE_START_STEP + 1) / max(c.CLDICE_WARMUP_STEPS, 1))
            self.mu = c.CLDICE_MU * r          # ramp, not a step change
        else:
            self.mu = 0.0
    def forward(self, logits, target, relw):
        c = self.cfg
        logits = logits.float()                                   # fp32, outside autocast
        valid = target != c.IGNORE_INDEX
        ce_map = F.cross_entropy(logits, target.clamp(max=c.NUM_CLASSES - 1), weight=self.cw,
                                 reduction="none", label_smoothing=c.LABEL_SMOOTH)
        ce = (ce_map * relw * valid).sum() / (relw * valid).sum().clamp(min=1.0)
        probs = logits.softmax(1)
        keep = valid & (relw >= c.CSA_LOVASZ_THRESH)              # hard mask, not a weight
        if keep.sum() > 0 and self.lam > 0:
            pf = probs.permute(0, 2, 3, 1)[keep]
            lv = lovasz_softmax_flat(pf, target[keep])
        else:
            lv = logits.sum() * 0.0
        cl = logits.sum() * 0.0
        if self.mu > 0:
            r = c.CLDICE_RES
            pr = F.adaptive_avg_pool2d(probs, r)
            terms = []
            for ci in c.CLDICE_CLASSES:
                gt = ((target == ci) & valid).float().unsqueeze(1)
                gt = F.adaptive_avg_pool2d(gt, r)
                present = (gt.sum((1, 2, 3)) > 0)
                if present.any():
                    terms.append(soft_cldice(pr[:, ci:ci + 1][present], gt[present], c.CLDICE_ITERS).mean())
            if terms:
                cl = torch.stack(terms).mean()
        total = ce + self.lam * lv + self.mu * cl
        return total, dict(ce=float(ce), lovasz=float(lv), cldice=float(cl), lam=self.lam, mu=self.mu)

criterion = GeoLoss(cfg, CLASS_W_T).to(DEVICE)
print("loss ready | lovasz lambda target", cfg.LOVASZ_LAMBDA, "| clDice mu", cfg.CLDICE_MU,
      "on", [CLASS_NAMES[i] for i in cfg.CLDICE_CLASSES])

## 12. Model — TC-SegFormer

**Unchanged from the Kathmandu notebook** apart from `num_labels = 7`. That is deliberate: any
architectural tweak made for LoveDA would turn the benchmark from "our method transfers" into "a
LoveDA-specific variant of our method scores X", which is a much weaker and much less interesting
claim.

```
image 512 ──> MiT encoder ──> 4 stages ──> linear_c + linear_fuse ──> fused (D, 128, 128)
   │                                                                     │
   │                                                        1x1 proj to 64 ch
   │                                                                     │
   └──> detail path (3->8->16->32, stride 2, GroupNorm) ──> (32, 256, 256)
                                                                         │
                                          concat ──> fuse 3x3 ──> refine 1x1 ──> residual @ H/2
```

The ADE20K-pretrained checkpoint is used as **initialisation only**, which is its correct role and
worth one sentence in the paper — it is not an ADE20K benchmark result.

With `USE_DETAIL=False` the forward pass returns the vanilla SegFormer logits on exactly the code
path a stock `SegformerForSemanticSegmentation` would take, so the control is a true control.

In [ ]:
from transformers import SegformerConfig, SegformerForSemanticSegmentation

class DetailPath(nn.Module):
    def __init__(self, chs):
        super().__init__()
        c1, c2, c3 = chs
        def blk(i, o, s): return nn.Sequential(nn.Conv2d(i, o, 3, s, 1, bias=False),
                                               nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
        self.net = nn.Sequential(blk(3, c1, 2), blk(c1, c2, 1), blk(c2, c3, 1))
        self.out_ch = c3
    def forward(self, x): return self.net(x)

class TCSegFormer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        conf = SegformerConfig.from_pretrained(cfg.BACKBONE)
        conf.num_labels = cfg.NUM_CLASSES
        conf.drop_path_rate = cfg.DROP_PATH
        base = SegformerForSemanticSegmentation.from_pretrained(
            cfg.BACKBONE, config=conf, ignore_mismatched_sizes=True)
        self.encoder = base.segformer
        self.head = base.decode_head
        D = conf.decoder_hidden_size
        self.use_detail = cfg.USE_DETAIL
        if self.use_detail:
            self.proj = nn.Conv2d(D, cfg.FUSE_CH, 1)
            self.detail = DetailPath(cfg.DETAIL_CH)
            self.gate = nn.Parameter(torch.tensor(float(cfg.DETAIL_GATE_INIT)))
            self.fuse = nn.Sequential(
                nn.Conv2d(cfg.FUSE_CH + self.detail.out_ch, cfg.FUSE_CH, 3, 1, 1, bias=False),
                nn.GroupNorm(8, cfg.FUSE_CH), nn.ReLU(inplace=True))
            self.refine = nn.Conv2d(cfg.FUSE_CH, cfg.NUM_CLASSES, 1)
            nn.init.normal_(self.refine.weight, std=1e-4); nn.init.zeros_(self.refine.bias)

    def fused_feature(self, hidden_states):
        bs = hidden_states[-1].shape[0]
        outs = []
        for hs, mlp in zip(hidden_states, self.head.linear_c):
            h, w = hs.shape[2], hs.shape[3]
            x = mlp(hs).permute(0, 2, 1).reshape(bs, -1, h, w)
            x = F.interpolate(x, size=hidden_states[0].shape[2:], mode="bilinear", align_corners=False)
            outs.append(x)
        x = self.head.linear_fuse(torch.cat(outs[::-1], dim=1))
        x = self.head.batch_norm(x)
        x = self.head.activation(x)
        return self.head.dropout(x)

    def forward(self, x):
        H, W = x.shape[-2:]
        hs = self.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
        fused = self.fused_feature(hs)
        base = self.head.classifier(fused)
        logits = F.interpolate(base, size=(H, W), mode="bilinear", align_corners=False)
        if not self.use_detail:
            return logits
        f = F.interpolate(self.proj(fused), size=(H // 2, W // 2), mode="bilinear", align_corners=False)
        f = torch.cat([f, self.gate * self.detail(x)], dim=1)
        r = self.refine(self.fuse(f))
        return logits + F.interpolate(r, size=(H, W), mode="bilinear", align_corners=False)

model = TCSegFormer(cfg).to(DEVICE).to(memory_format=torch.channels_last)
NEW_PREFIX = ("proj", "detail", "fuse", "refine", "gate")
n_all = sum(p.numel() for p in model.parameters())
n_new = sum(p.numel() for n, p in model.named_parameters() if n.split(".")[0] in NEW_PREFIX)
print(f"params total {n_all/1e6:.2f} M | new modules {n_new/1e6:.3f} M "
      f"(+{100*n_new/max(n_all-n_new,1):.2f}%)")
with torch.no_grad(), torch.cuda.amp.autocast():
    o512 = model(torch.zeros(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=DEVICE).to(memory_format=torch.channels_last))
    o1024 = model(torch.zeros(1, 3, cfg.EVAL_SIZE, cfg.EVAL_SIZE, device=DEVICE).to(memory_format=torch.channels_last))
print("forward ok -> train", tuple(o512.shape), "| eval", tuple(o1024.shape))
torch.cuda.empty_cache()

## 13. Optimizer, schedule, EMA

Unchanged. 10x differential LR: ADE20K-pretrained encoder at `LR_ENCODER`, the pretrained decode head
at `LR_HEAD`, and the **randomly-initialised new modules at `LR_NEW`** — a stride-1 conv on the raw
512x512 image sees far higher-variance input than the pretrained encoder does. Norm and bias
parameters are excluded from weight decay.

Warmup 500 steps then cosine over `MAX_STEPS`. EMA decay 0.999, and **validation runs on the EMA
weights**, matching the Kathmandu protocol.

In [ ]:
def param_groups(model, cfg):
    g = {k: [] for k in ("enc_d", "enc_n", "head_d", "head_n", "new_d", "new_n")}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        root = n.split(".")[0]
        tier = "new_" if root in NEW_PREFIX else ("enc_" if root == "encoder" else "head_")
        no_decay = p.ndim <= 1 or "norm" in n.lower() or n.endswith(".bias") or n == "gate"
        g[tier + ("n" if no_decay else "d")].append(p)
    wd = cfg.WEIGHT_DECAY
    return [dict(params=g["enc_d"],  lr=cfg.LR_ENCODER, weight_decay=wd),
            dict(params=g["enc_n"],  lr=cfg.LR_ENCODER, weight_decay=0.0),
            dict(params=g["head_d"], lr=cfg.LR_HEAD,    weight_decay=wd),
            dict(params=g["head_n"], lr=cfg.LR_HEAD,    weight_decay=0.0),
            dict(params=g["new_d"],  lr=cfg.LR_NEW,     weight_decay=wd),
            dict(params=g["new_n"],  lr=cfg.LR_NEW,     weight_decay=0.0)]

optimizer = torch.optim.AdamW(param_groups(model, cfg))
for nm, grp in zip(["enc_d", "enc_n", "head_d", "head_n", "new_d", "new_n"], optimizer.param_groups):
    print(f"  {nm:<7} lr {grp['lr']:.1e}  wd {grp['weight_decay']}  tensors {len(grp['params'])}")
scaler = torch.cuda.amp.GradScaler()
TOTAL_STEPS = cfg.MAX_STEPS

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return (step + 1) / cfg.WARMUP_STEPS
    p = (step - cfg.WARMUP_STEPS) / max(1, TOTAL_STEPS - cfg.WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items() if v.dtype.is_floating_point}
        self.backup = None
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
    def apply_to(self, model):
        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items() if k in self.shadow}
        model.load_state_dict({k: (self.shadow[k].to(v.dtype) if k in self.shadow else v)
                               for k, v in model.state_dict().items()}, strict=True)
    def restore(self, model):
        if self.backup is None: return
        sd = model.state_dict(); sd.update(self.backup); model.load_state_dict(sd, strict=True)
        self.backup = None

ema = EMA(model, cfg.EMA_DECAY) if cfg.EMA else None
print(f"total {TOTAL_STEPS} steps | eval every {cfg.EVAL_EVERY} | EMA {bool(ema)}")

## 14. Metrics

One 7x7 confusion matrix accumulated on-GPU with `bincount`, ignore pixels dropped before
accumulation. IoU, F1, OA, fwIoU, kappa and producer's/user's accuracy are all derived from it, so
nothing can drift out of sync.

**Everything the LoveDA table needs comes out of this one function**: `mIoU` is the headline,
`IoU[...]` gives the seven per-class values that every published LoveDA table reports, and
`UA`/`PA` tell you whether a low Barren IoU is under-prediction or over-prediction — which IoU and F1
alone cannot, since on this metric set F1 is algebraically `2*IoU/(1+IoU)` and carries no extra
information.

In [ ]:
@torch.no_grad()
def update_cm(cm, pred, target, mask=None):
    v = target != cfg.IGNORE_INDEX
    if mask is not None: v = v & mask
    if v.sum() == 0: return cm
    t, p = target[v].reshape(-1), pred[v].reshape(-1)
    cm += torch.bincount(t * cfg.NUM_CLASSES + p, minlength=cfg.NUM_CLASSES ** 2).reshape(cfg.NUM_CLASSES, -1)
    return cm

def cm_metrics(cm):
    cm = cm.double().cpu().numpy()
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = tp + fp + fn
    iou = np.where(den > 0, tp / np.maximum(den, 1e-9), np.nan)
    f1 = np.where((2 * tp + fp + fn) > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), np.nan)
    prec = np.where((tp + fp) > 0, tp / np.maximum(tp + fp, 1e-9), np.nan)     # user's accuracy
    rec = np.where((tp + fn) > 0, tp / np.maximum(tp + fn, 1e-9), np.nan)      # producer's accuracy
    N = cm.sum(); oa = tp.sum() / max(N, 1e-9)
    freq = cm.sum(1) / max(N, 1e-9)
    fwiou = float(np.nansum(freq * iou))
    pe = float((cm.sum(0) * cm.sum(1)).sum() / max(N * N, 1e-9))
    kappa = (oa - pe) / max(1 - pe, 1e-9)
    return dict(mIoU=float(np.nanmean(iou)), mF1=float(np.nanmean(f1)), OA=float(oa),
                fwIoU=fwiou, kappa=float(kappa),
                IoU={n: float(v) for n, v in zip(CLASS_NAMES, iou)},
                F1={n: float(v) for n, v in zip(CLASS_NAMES, f1)},
                UA={n: float(v) for n, v in zip(CLASS_NAMES, prec)},
                PA={n: float(v) for n, v in zip(CLASS_NAMES, rec)},
                cm_norm=(cm / np.maximum(cm.sum(1, keepdims=True), 1e-9)).tolist())

def new_cm(): return torch.zeros(cfg.NUM_CLASSES, cfg.NUM_CLASSES, dtype=torch.long, device=DEVICE)
print("metrics ready | 7 classes:", CLASS_NAMES)

## 15. Efficiency profile

Params, GFLOPs at **both** 512 (training crop) and 1024 (evaluation tile), peak training VRAM,
wall-clock, and inference FPS. The published LoveDA tables quote complexity and speed at 1024x1024
input, so the 1024 column is the one that belongs next to them; the 512 column is what the Kathmandu
table used, and reporting both keeps the two tables mutually readable.

T4-specific facts worth one sentence in the paper: sm_75 has no bf16 and no FlashAttention-2, so this
is fp16 + GradScaler with standard attention. SegFormer's sequence-reduction attention keeps stage 1
tractable, so attention is not the scaling bottleneck — stage-1 MLP activations are.

In [ ]:
def efficiency_report(model, cfg):
    rep = {"params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    for size, key in ((cfg.IMG_SIZE, "GFLOPs_512"), (cfg.EVAL_SIZE, "GFLOPs_1024")):
        try:
            from fvcore.nn import FlopCountAnalysis
            m = model.eval()
            fa = FlopCountAnalysis(m, torch.zeros(1, 3, size, size, device=DEVICE))
            fa.unsupported_ops_warnings(False); fa.uncalled_modules_warnings(False)
            rep[key] = fa.total() / 1e9
        except Exception as e:
            rep[key] = None; rep[f"flops_error_{size}"] = str(e)[:120]
    model.eval()
    for size, key in ((cfg.IMG_SIZE, "infer_FPS_bs1_512"), (cfg.EVAL_SIZE, "infer_FPS_bs1_1024")):
        x = torch.zeros(1, 3, size, size, device=DEVICE).to(memory_format=torch.channels_last)
        with torch.no_grad(), torch.cuda.amp.autocast():
            for _ in range(20): model(x)
            torch.cuda.synchronize(); t0 = time.time()
            for _ in range(50): model(x)
            torch.cuda.synchronize()
        rep[key] = 50 / (time.time() - t0)
    model.train(); torch.cuda.empty_cache()
    return rep

EFF = efficiency_report(model, cfg)
print(json.dumps(EFF, indent=1))

## 16. Hugging Face persistence — Pattern A (one repo per run)

`/kaggle/working` is temporary and a 20k-step run can straddle two sessions, so checkpoints live on
the Hub. One repo per `RUN_TAG` means the TC-SegFormer run and the SegFormer-B0 control never
overwrite each other, and either is downloadable for standalone inference later.

* **`best.pt`** — weights + config + metrics, pushed on every selection-mIoU improvement.
* **`last.pt`** — full optimizer / scaler / scheduler / EMA / global-step state, pushed every
  `PUSH_LAST_EVERY` evaluations. This is what a new session resumes from.
* **`stats.npz`** — the section 6 scan, so a resumed session skips 90 s of mask decoding.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", e)
api = HfApi(token=HF_TOKEN) if HF_TOKEN else None
if api and cfg.HF_PUSH:
    create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_upload(local, name):
    if not (api and cfg.HF_PUSH): return
    try:
        api.upload_file(path_or_fileobj=str(local), path_in_repo=name, repo_id=HF_REPO)
    except Exception as e:
        print("HF upload failed:", str(e)[:160])

def hf_try_download(name):
    if not api: return None
    try:
        return hf_hub_download(HF_REPO, name, token=HF_TOKEN)
    except Exception:
        return None

def write_model_card(metrics, step):
    card = f"""---
tags: [semantic-segmentation, remote-sensing, segformer, loveda, geonusaf]
datasets: [LoveDA]
---
# GeoNUSAF {cfg.PRESET} -- LoveDA benchmark ({cfg.DOMAIN_FILTER})

LoveDA land-cover segmentation, 7 classes, ignore_index=255, 512x512 crops @ 0.3 m/px,
evaluated at 1024x1024 on the official validation split.

| | |
|---|---|
| backbone | `{cfg.BACKBONE}` (ADE20K init only) |
| split | official LoveDA Train -> Val |
| budget | {cfg.MAX_STEPS} steps, batch {cfg.BATCH_SIZE}, single seed {cfg.SEED} |
| best step | {step} |
| detail path | {cfg.USE_DETAIL} |
| CSA reweighting | {cfg.CSA_ENABLE} (tau={list(cfg.CSA_TAU)}, w_min={cfg.CSA_WMIN}) |
| soft-clDice | {cfg.CLDICE_ENABLE} (mu={cfg.CLDICE_MU}, classes Road/Water) |
| balanced sampler | {cfg.BALANCED_SAMPLER} |

**Not a leaderboard entry.** This is a fixed-budget controlled comparison against the
SegFormer-B0 baseline trained identically, not the 80k-iteration schedule used by
published LoveDA results.

## Validation (official Val split)
mIoU **{metrics['mIoU']:.4f}** | mF1 {metrics['mF1']:.4f} | OA {metrics['OA']:.4f} | kappa {metrics['kappa']:.4f}

| class | IoU | UA (prec) | PA (rec) |
|---|---|---|---|
""" + "\n".join(f"| {n} | {metrics['IoU'][n]:.4f} | {metrics['UA'][n]:.4f} | {metrics['PA'][n]:.4f} |" for n in CLASS_NAMES)
    p = Path(cfg.OUT_DIR) / "README.md"; p.write_text(card); hf_upload(p, "README.md")

if STATS_P.exists(): hf_upload(STATS_P, "stats.npz")
print("HF ready:", HF_REPO if api else "(disabled)")

## 17. Weights & Biases

`resume="allow"` with a deterministic run id derived from `RUN_TAG`, so a resumed session continues
the same run instead of starting a second one next to it. Logging is keyed on `global_step`, which
makes the TC-SegFormer and SegFormer-B0 curves directly overlayable in the W&B UI — that overlay is
the figure the paper wants.

In [ ]:
WB = None
if cfg.WANDB:
    try:
        import wandb
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
        WB = wandb.init(project=cfg.WANDB_PROJECT, name=RUN_TAG, id=RUN_TAG.replace("/", "-"),
                        resume="allow", config={**asdict(cfg), **EFF})
        print("wandb:", WB.url)
    except Exception as e:
        print("wandb disabled:", str(e)[:160]); WB = None

## 18. Train and validate loops

`scaler.unscale_(optimizer)` runs **before** `clip_grad_norm_`. Clipping loss-scaled gradients makes
the 1.0 threshold meaningless — this is the exact bug that lets 6e5 gradient norms through. The
returned pre-clip norm is logged as the direct check against that explosion.

Training is a step loop over an **infinite** iterator, so there is no epoch boundary to resume across;
`global_step` alone is sufficient state. The dataset's augmentation epoch counter is advanced each
time the sampler is exhausted so the augmentation stream keeps changing.

`validate()` takes a loader so the same code serves both the cheap selection subset and the full
1,669-tile report in section 20. It accumulates a full confusion matrix plus **one per domain**, which
is what makes the urban/rural table free.

In [ ]:
@torch.no_grad()
def slide_inference(net, img, crop, stride, n_classes):
    # mmseg-style sliding window. Returns averaged softmax probabilities at full resolution.
    B, _, H, W = img.shape
    hg = max(H - crop + stride - 1, 0) // stride + 1
    wg = max(W - crop + stride - 1, 0) // stride + 1
    acc = img.new_zeros((B, n_classes, H, W))
    cnt = img.new_zeros((B, 1, H, W))
    for hi in range(hg):
        for wi in range(wg):
            y2 = min(hi * stride + crop, H); x2 = min(wi * stride + crop, W)
            y1 = max(y2 - crop, 0); x1 = max(x2 - crop, 0)
            with torch.cuda.amp.autocast():
                lg = net(img[:, :, y1:y2, x1:x2])
            acc[:, :, y1:y2, x1:x2] += lg.float().softmax(1)
            cnt[:, :, y1:y2, x1:x2] += 1
    return acc / cnt

def forward_eval(net, x, stride):
    # Returns log-probabilities. softmax(log p) == p, so the loss and argmax are unchanged.
    if not cfg.SLIDE_INFER:
        with torch.cuda.amp.autocast():
            return net(x).float()
    return slide_inference(net, x, cfg.SLIDE_CROP, stride, cfg.NUM_CLASSES).clamp_min(1e-8).log()

def infinite(dl, ds):
    ep = 0
    while True:
        ds.set_epoch(ep)
        for b in dl:
            yield b
        ep += 1

train_iter = infinite(train_dl, train_ds)

def train_n_steps(n, global_step):
    model.train()
    agg = dict(loss=0.0, ce=0.0, lovasz=0.0, cldice=0.0, gnorm=0.0); nb = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(range(n), desc=f"train -> {global_step + n}", leave=False)
    for i in pbar:
        x, y, w = next(train_iter)
        criterion.set_step(global_step)
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True); w = w.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits = model(x)
        loss, parts = criterion(logits, y, w)                 # fp32, outside autocast
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()
        if (i + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)                        # MUST precede clipping
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            if ema: ema.update(model)
            agg["gnorm"] += float(gn)
        agg["loss"] += float(loss); agg["ce"] += parts["ce"]
        agg["lovasz"] += parts["lovasz"]; agg["cldice"] += parts["cldice"]; nb += 1
        global_step += 1
        if i % 50 == 0:
            pbar.set_postfix(loss=f"{float(loss):.3f}", gn=f'{agg["gnorm"]/max(1,i+1):.1f}')
    k = max(nb, 1)
    return global_step, {"train_loss": agg["loss"]/k, "train_ce": agg["ce"]/k,
                         "train_lovasz": agg["lovasz"]/k, "train_cldice": agg["cldice"]/k,
                         "grad_norm": agg["gnorm"]/k, "lovasz_lambda": criterion.lam,
                         "cldice_mu": criterion.mu,
                         "lr_head": optimizer.param_groups[2]["lr"],
                         "gate": float(model.gate.detach()) if cfg.USE_DETAIL else 0.0}

@torch.no_grad()
def validate(loader, rows, desc="val", per_image=False, stride=None):
    stride = stride or cfg.SEL_SLIDE_STRIDE
    if ema: ema.apply_to(model)
    model.eval()
    cms = {"all": new_cm(), "core": new_cm()}
    for d in sorted({r["domain"] for r in rows}):
        cms[d] = new_cm()
    vloss, vce, vlov, nb, seen = 0.0, 0.0, 0.0, 0, 0
    pim = []
    for x, y, w in tqdm(loader, desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True); w = w.to(DEVICE, non_blocking=True)
        logits = forward_eval(model, x, stride)
        loss, vparts = criterion(logits, y, w)
        vloss += float(loss); vce += vparts["ce"]; vlov += vparts["lovasz"]; nb += 1
        pred = logits.float().argmax(1)
        update_cm(cms["all"], pred, y)
        update_cm(cms["core"], pred, y, mask=(w >= cfg.CORE_THRESH))
        for k in range(pred.shape[0]):
            dom = rows[seen + k]["domain"]
            update_cm(cms[dom], pred[k:k + 1], y[k:k + 1])
            if per_image:
                c1 = new_cm(); update_cm(c1, pred[k:k + 1], y[k:k + 1])
                pim.append({"i": seen + k, "stem": rows[seen + k]["stem"],
                            "domain": dom, "mIoU": cm_metrics(c1)["mIoU"]})
        seen += pred.shape[0]
    if ema: ema.restore(model)
    model.train()
    out = {k: cm_metrics(v) for k, v in cms.items()}
    n = max(nb, 1)
    # val_loss changes COMPOSITION when clDice switches on, so it is not comparable across the
    # whole run. val_ce and val_ce_lovasz are fixed-composition -- read those, and sel_mIoU.
    return {"val_loss": vloss / n, "val_ce": vce / n, "val_lovasz": vlov / n,
            "val_ce_lovasz": vce / n + cfg.LOVASZ_LAMBDA * vlov / n}, out, cms["all"], pim
print("loops ready")

## 19. Main loop with automatic resume and a session-time guard

Early stopping is on **selection mIoU**, not val loss — the loss is class-weighted and CSA-weighted
and moves for reasons mIoU does not care about. Patience is counted in *evaluations* (6 x 1000 steps
= 6k steps of no meaningful improvement).

**Session guard.** `SESSION_HOURS = 10.5` stops the loop cleanly and pushes `last.pt` before Kaggle
kills the kernel at 12 h. This is the exact failure that cut the stage-2 run short — a hard kill loses
whatever was not uploaded. Re-run the notebook top to bottom in a new session and it resumes from the
stored `global_step`, optimizer, scaler, scheduler and EMA state. **The schedule is never shortened to
fit a session**, because a truncated cosine is a different experiment.

**Architecture guard on resume.** One repo per run means a checkpoint written by an older module
layout can still be sitting there. The `ARCH_VERSION` check happens *before* any `load_state_dict`,
because a partial copy leaves the model in a mixed state that is worse than starting fresh.

In [ ]:
CKPT_DIR = Path(cfg.OUT_DIR); CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST, BEST = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"

global_step, best_miou, best_step, bad, n_eval = 0, -1.0, -1, 0, 0
ref_miou = -1.0
history = []

def _compatible(sd):
    cur = model.state_dict()
    if set(cur) != set(sd):
        return False, (f"key sets differ (checkpoint-only {sorted(set(sd)-set(cur))[:4]}, "
                       f"model-only {sorted(set(cur)-set(sd))[:4]})")
    for k in cur:
        if tuple(cur[k].shape) != tuple(sd[k].shape):
            return False, f"shape mismatch at {k}: {tuple(sd[k].shape)} vs {tuple(cur[k].shape)}"
    return True, ""

def _purge_stale(reason):
    print(f"!! IGNORING the remote checkpoint -- {reason}")
    print(f"   This run starts from scratch; stale artifacts are being deleted from {HF_REPO}.")
    if api and cfg.HF_PUSH:
        for f in ("last.pt", "best.pt"):
            try: api.delete_file(path_in_repo=f, repo_id=HF_REPO)
            except Exception: pass

remote_last = hf_try_download("last.pt") if cfg.RESUME else None
if remote_last:
    ck = torch.load(remote_last, map_location="cpu", weights_only=False)
    ok, why = _compatible(ck["model"])
    if ck.get("arch") != cfg.ARCH_VERSION:
        _purge_stale(f'architecture {ck.get("arch", "<pre-versioning>")!r} != {cfg.ARCH_VERSION!r}')
    elif ck.get("preset") != cfg.PRESET:
        _purge_stale(f'preset {ck.get("preset")!r} != {cfg.PRESET!r}')
    elif not ok:
        _purge_stale(why)
    else:
        model.load_state_dict(ck["model"], strict=True); optimizer.load_state_dict(ck["optim"])
        scaler.load_state_dict(ck["scaler"]); scheduler.load_state_dict(ck["sched"])
        if ema and ck.get("ema"): ema.shadow = {k: v.to(DEVICE) for k, v in ck["ema"].items()}
        global_step = ck["global_step"]; best_miou = ck["best_miou"]; best_step = ck["best_step"]
        bad = ck.get("bad", 0); n_eval = ck.get("n_eval", 0); history = ck.get("history", [])
        ref_miou = ck.get("ref_miou", best_miou)
        print(f"resumed from HF at step {global_step} (best mIoU {best_miou:.4f} @ {best_step}, "
              f"patience ref {ref_miou:.4f}, bad {bad})")
elif cfg.RESUME:
    print("no remote checkpoint -- starting fresh")
else:
    print("RESUME=False -- starting fresh, remote checkpoint ignored")

print(f"early stopping: patience {cfg.PATIENCE} evaluations, min_delta {cfg.MIN_DELTA} "
      f"| horizon {cfg.MAX_STEPS} steps | session cap {cfg.SESSION_HOURS} h")

def save_last():
    torch.save({"model": model.state_dict(), "optim": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "sched": scheduler.state_dict(),
                "ema": (ema.shadow if ema else None), "global_step": global_step, "bad": bad,
                "n_eval": n_eval, "best_miou": best_miou, "best_step": best_step,
                "ref_miou": ref_miou, "history": history, "cfg": asdict(cfg),
                "arch": cfg.ARCH_VERSION, "preset": cfg.PRESET}, LAST)
    hf_upload(LAST, "last.pt")

t_start = time.time()
stop_reason = "horizon reached"
while global_step < cfg.MAX_STEPS:
    n = min(cfg.EVAL_EVERY, cfg.MAX_STEPS - global_step)
    global_step, tr = train_n_steps(n, global_step)
    vl, M, cm, _ = validate(sel_dl, SEL_ROWS, desc=f"select @{global_step}",
                            stride=cfg.SEL_SLIDE_STRIDE)
    n_eval += 1
    mfull = M["all"]
    row = {"step": global_step, **tr, **vl,
           "gen_gap_ce": vl["val_ce"] - tr["train_ce"],
           "sel_mIoU": mfull["mIoU"], "sel_mF1": mfull["mF1"], "sel_OA": mfull["OA"],
           "sel_kappa": mfull["kappa"], "sel_fwIoU": mfull["fwIoU"],
           **{f"IoU/{k}": v for k, v in mfull["IoU"].items()},
           **{f"mIoU/{d}": M[d]["mIoU"] for d in M if d not in ("all", "core")}}
    history.append(row)
    if WB: WB.log(row, step=global_step)
    dom_s = " ".join(f"{d[:1]}{M[d]['mIoU']:.3f}" for d in sorted(M) if d not in ("all", "core"))
    print(f's{global_step:06d} CE {tr["train_ce"]:.3f}/{vl["val_ce"]:.3f} '
          f'(gap {row["gen_gap_ce"]:+.2f}) lov {tr["train_lovasz"]:.3f} lam {tr["lovasz_lambda"]:.2f} | '
          f'selmIoU {mfull["mIoU"]:.4f} [{dom_s}] | gn {tr["grad_norm"]:.2f} | gate {tr["gate"]:.3f} | '
          f'road {mfull["IoU"]["Road"]:.3f} water {mfull["IoU"]["Water"]:.3f} '
          f'barren {mfull["IoU"]["Barren"]:.3f}')

    m = mfull["mIoU"]
    if m > ref_miou + cfg.MIN_DELTA:
        ref_miou, bad = m, 0
    else:
        bad += 1

    if m > best_miou:
        best_miou, best_step = m, global_step
        sd = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if ema:
            for k, v in ema.shadow.items():
                sd[k] = v.to(sd[k].dtype)
        torch.save({"model": {k: v.cpu() for k, v in sd.items()}, "cfg": asdict(cfg),
                    "arch": cfg.ARCH_VERSION, "preset": cfg.PRESET, "class_names": CLASS_NAMES,
                    "metrics_sel": mfull, "global_step": global_step, "efficiency": EFF}, BEST)
        hf_upload(BEST, "best.pt"); write_model_card(mfull, global_step)

    elapsed = (time.time() - t_start) / 3600
    if n_eval % cfg.PUSH_LAST_EVERY == 0 or bad >= cfg.PATIENCE or global_step >= cfg.MAX_STEPS:
        save_last()
    if bad >= cfg.PATIENCE:
        stop_reason = f"early stop (patience {cfg.PATIENCE})"; break
    if elapsed > cfg.SESSION_HOURS:
        save_last()
        stop_reason = (f"SESSION CAP {cfg.SESSION_HOURS} h reached at step {global_step}/{cfg.MAX_STEPS}. "
                       f"Re-run this notebook in a new session -- it resumes from last.pt.")
        break

TRAIN_HOURS = (time.time() - t_start) / 3600
print(f"\n{stop_reason}")
print(f"training {TRAIN_HOURS:.2f} h this session | best selection mIoU {best_miou:.4f} @ step {best_step}")
INCOMPLETE = global_step < cfg.MAX_STEPS and "SESSION CAP" in stop_reason
if INCOMPLETE:
    print("!! The cosine schedule did NOT finish. Do not report this run -- resume it first.")

## 20. Final evaluation — full validation split, urban/rural, clDice

The best checkpoint is reloaded and evaluated once on **all 1,669 validation tiles at 1024x1024**.
This is the number that goes in the paper; the per-evaluation `sel_mIoU` above is a selection signal
measured on a 400-tile subset and must never be reported as the result.

Four things come out of this cell:

* **Full protocol** — every valid pixel, 7-class mIoU plus per-class IoU/F1/UA/PA. The published-table row.
* **Urban and Rural separately** — LoveDA's defining axis. The gap between them is the domain-shift
  number, and it costs nothing to produce.
* **clDice for Road and Water** using true skeletonisation. Not a LoveDA convention, so it belongs in a
  supplementary table — but it is the metric that directly tests the detail path's reason for existing.
* **Core protocol** — pixels with `relw >= CORE_THRESH`. Diagnostic only. LoveDA's annotations are not
  coarse in the way Kathmandu's are, and no published number uses this, so **do not put it in the
  comparison table**.

In [ ]:
from skimage.morphology import skeletonize

def cldice_pair(pred_b, gt_b, eps=1e-6):
    if pred_b.sum() == 0 and gt_b.sum() == 0: return None
    sp = skeletonize(pred_b) if pred_b.any() else np.zeros_like(pred_b)
    sg = skeletonize(gt_b) if gt_b.any() else np.zeros_like(gt_b)
    tprec = (sp & gt_b).sum() / (sp.sum() + eps) if sp.sum() else 0.0
    tsens = (sg & pred_b).sum() / (sg.sum() + eps) if sg.sum() else 0.0
    return 0.0 if (tprec + tsens) == 0 else float(2 * tprec * tsens / (tprec + tsens))

ck = torch.load(hf_try_download("best.pt") or BEST, map_location="cpu", weights_only=False)
model.load_state_dict({k: v.to(DEVICE) for k, v in ck["model"].items()}, strict=True)
model.eval()
print(f'evaluating best.pt from step {ck.get("global_step")} on {len(VAL_ROWS)} val tiles @ {cfg.EVAL_SIZE}px')

print(f"inference: {'sliding window ' + str(cfg.SLIDE_CROP) + '/' + str(cfg.SLIDE_STRIDE) if cfg.SLIDE_INFER else 'whole tile'}")
cms = {"all": new_cm(), "core": new_cm()}
for d in sorted({r["domain"] for r in VAL_ROWS}): cms[d] = new_cm()
cld = {ci: [] for ci in cfg.CLDICE_CLASSES}
per_image, seen = [], 0
peak_vram = torch.cuda.max_memory_allocated() / 2 ** 30

with torch.no_grad():
    for x, y, w in tqdm(val_dl, desc="final eval"):
        x = x.to(DEVICE).to(memory_format=torch.channels_last); y = y.to(DEVICE); w = w.to(DEVICE)
        logits = forward_eval(model, x, cfg.SLIDE_STRIDE)
        pred = logits.argmax(1)
        update_cm(cms["all"], pred, y)
        update_cm(cms["core"], pred, y, mask=(w >= cfg.CORE_THRESH))
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        for k in range(pn.shape[0]):
            rec = VAL_ROWS[seen + k]
            update_cm(cms[rec["domain"]], pred[k:k + 1], y[k:k + 1])
            c1 = new_cm(); update_cm(c1, pred[k:k + 1], y[k:k + 1])
            per_image.append({"i": seen + k, "stem": rec["stem"], "domain": rec["domain"],
                              "mIoU": cm_metrics(c1)["mIoU"]})
            for ci in cfg.CLDICE_CLASSES:
                v = cldice_pair(pn[k] == ci, yn[k] == ci)
                if v is not None: cld[ci].append(v)
        seen += pn.shape[0]

M_FULL = cm_metrics(cms["all"])
M_CORE = cm_metrics(cms["core"])
M_DOM = {d: cm_metrics(cms[d]) for d in cms if d not in ("all", "core")}
CLDICE = {CLASS_NAMES[ci]: (float(np.mean(cld[ci])) if cld[ci] else float("nan")) for ci in cfg.CLDICE_CLASSES}

print(f'\nFULL VAL ({len(VAL_ROWS)} tiles)')
print(f'  mIoU {M_FULL["mIoU"]:.4f} | mF1 {M_FULL["mF1"]:.4f} | OA {M_FULL["OA"]:.4f} | '
      f'fwIoU {M_FULL["fwIoU"]:.4f} | kappa {M_FULL["kappa"]:.4f}')
for d in sorted(M_DOM):
    print(f'  {d:<6} mIoU {M_DOM[d]["mIoU"]:.4f}')
if len(M_DOM) == 2:
    a, b = sorted(M_DOM)
    print(f'  DOMAIN GAP ({a} - {b}) = {M_DOM[a]["mIoU"] - M_DOM[b]["mIoU"]:+.4f} mIoU')
print(f'  core mIoU {M_CORE["mIoU"]:.4f} (diagnostic only -- NOT comparable to published numbers)')
print("  clDice:", {k: round(v, 4) for k, v in CLDICE.items()})

print(f'\n{"class":<13}{"IoU":>8}{"F1":>8}{"UA":>8}{"PA":>8}' +
      "".join(f'{d[:5]+" IoU":>10}' for d in sorted(M_DOM)))
for n in CLASS_NAMES:
    print(f'{n:<13}{M_FULL["IoU"][n]:>8.4f}{M_FULL["F1"][n]:>8.4f}{M_FULL["UA"][n]:>8.4f}'
          f'{M_FULL["PA"][n]:>8.4f}' + "".join(f'{M_DOM[d]["IoU"][n]:>10.4f}' for d in sorted(M_DOM)))

## 21. Results persistence

CSV and JSON, both pushed to the Hub, so building the comparison table later depends on nothing
surviving in `/kaggle/working`. The flat CSV row is shaped so that the TC-SegFormer row and the
SegFormer-B0 row concatenate directly in section 22.

`incomplete` is written into the record. A run that hit the session cap before finishing its cosine is
**not a result**, and the flag is there so a later session cannot accidentally table it.

In [ ]:
import pandas as pd
EFF["peak_train_VRAM_GB"] = peak_vram
EFF["train_hours_last_session"] = TRAIN_HOURS

RESULT = {"run": RUN_TAG, "preset": cfg.PRESET, "dataset": "LoveDA", "backbone": cfg.BACKBONE,
          "split": "official Train->Val", "domain_filter": cfg.DOMAIN_FILTER, "seed": cfg.SEED,
          "protocol": cfg.PROTOCOL,
          "eval": (f"slide {cfg.SLIDE_CROP}/{cfg.SLIDE_STRIDE}" if cfg.SLIDE_INFER
                   else f"whole {cfg.EVAL_SIZE}"),
          "scale_jitter": (list(cfg.SCALE_RANGE) if cfg.SCALE_JITTER else None),
          "csa_wmin": cfg.CSA_WMIN, "csa_lovasz_thresh": cfg.CSA_LOVASZ_THRESH,
          "steps": int(global_step), "max_steps": cfg.MAX_STEPS, "best_step": best_step,
          "incomplete": bool(INCOMPLETE),
          "n_train": len(TRAIN_ROWS), "n_val": len(VAL_ROWS),
          "ablation": {"detail": cfg.USE_DETAIL, "csa": cfg.CSA_ENABLE,
                       "cldice": cfg.CLDICE_ENABLE, "balanced_sampler": cfg.BALANCED_SAMPLER},
          "full": M_FULL, "core": M_CORE, "domains": M_DOM, "clDice": CLDICE, "efficiency": EFF,
          "class_weights": CLASS_W.tolist(), "beta": BETA}

jp = Path(cfg.OUT_DIR) / f"results_{RUN_TAG}.json"; json.dump(RESULT, open(jp, "w"), indent=2)
flat = {"run": RUN_TAG, "preset": cfg.PRESET, "protocol": cfg.PROTOCOL,
        "steps": int(global_step), "best_step": best_step,
        "incomplete": bool(INCOMPLETE),
        "mIoU": M_FULL["mIoU"], "mF1": M_FULL["mF1"], "OA": M_FULL["OA"],
        "fwIoU": M_FULL["fwIoU"], "kappa": M_FULL["kappa"], "core_mIoU": M_CORE["mIoU"],
        **{f"IoU_{k}": v for k, v in M_FULL["IoU"].items()},
        **{f"UA_{k}": v for k, v in M_FULL["UA"].items()},
        **{f"PA_{k}": v for k, v in M_FULL["PA"].items()},
        **{f"mIoU_{d}": M_DOM[d]["mIoU"] for d in M_DOM},
        **{f"clDice_{k}": v for k, v in CLDICE.items()},
        "params_M": EFF["params_M"], "GFLOPs_512": EFF.get("GFLOPs_512"),
        "GFLOPs_1024": EFF.get("GFLOPs_1024"), "FPS_1024": EFF.get("infer_FPS_bs1_1024"),
        "peak_VRAM_GB": peak_vram, "train_hours_last_session": TRAIN_HOURS}
cp = Path(cfg.OUT_DIR) / f"results_{RUN_TAG}.csv"; pd.DataFrame([flat]).to_csv(cp, index=False)
hp = Path(cfg.OUT_DIR) / f"history_{RUN_TAG}.csv"; pd.DataFrame(history).to_csv(hp, index=False)
pip = Path(cfg.OUT_DIR) / f"perimage_{RUN_TAG}.csv"; pd.DataFrame(per_image).to_csv(pip, index=False)
for p in (jp, cp, hp, pip):
    hf_upload(p, p.name)
if WB: WB.summary.update(flat)
print(pd.DataFrame([flat]).T.to_string())

## 22. Comparison table — our two runs against the published LoveDA baselines

This replaces the Kathmandu notebook's cross-fold aggregation, since there are no folds here.

**Read the caveat in the printout, and reproduce it verbatim in the paper caption.** The published
numbers below are **test-set** results from the official server, produced on 80k-iteration schedules
on datacentre GPUs. Ours are **validation-set** results from a 20k-step single-T4 budget. Putting them
in one table without that sentence is the kind of thing a reviewer catches immediately and it costs
you the paper's credibility on every other number too.

Two legitimate ways to make the rows directly comparable, in order of preference:

1. Run section 23's test export and submit to the official LoveDA server. Then your row is a test row.
2. If the server is no longer accepting submissions, keep the val numbers and label the column
   `Val` while the baselines are labelled `Test`, with a footnote.

The interesting comparison is not against row `UperNet (Swin-S)`. It is **`tcsegformer` vs
`segformer-b0` on the two rows you produced yourself**, which is the only pair in this table that
shares a budget.

In [ ]:
# published LoveDA TEST-set results, per-class IoU in CLASS_NAMES order
PUBLISHED = [
    # name,                    Bkg,  Bldg,  Road, Water, Barren, Forest, Agri,  mIoU
    ("U-Net (ResNet50)",      43.06, 52.74, 52.78, 73.08, 10.33, 43.05, 59.87, 47.84),
    ("DeepLabV3+ (ResNet50)", 42.97, 50.88, 52.02, 74.36, 10.40, 44.21, 58.53, 47.62),
    ("PSPNet (ResNet50)",     44.40, 52.13, 53.52, 76.50,  9.73, 44.07, 57.85, 48.31),
    ("FactSeg (ResNet50)",    42.60, 53.63, 52.79, 76.94, 16.20, 42.92, 57.50, 48.94),
    ("HRNet (W32)",           44.61, 55.34, 57.42, 73.96, 11.07, 45.25, 60.88, 49.79),
    ("UNetFormer (R18)",      44.70, 58.80, 54.90, 79.60, 20.10, 46.00, 62.50, 52.40),
    ("SegFormer-B5",          46.54, 57.46, 58.91, 80.09, 27.89, 46.14, 61.00, 54.01),
    ("UperNet (Swin-S)",      46.18, 59.97, 57.40, 81.20, 26.71, 47.21, 63.18, 54.55),
]

for tag in ("tcsegformer", "segformer-b0"):
    t = f"{tag}-loveda-{cfg.DOMAIN_FILTER.lower()}-{cfg.PROTOCOL}"
    if (Path(cfg.OUT_DIR) / f"results_{t}.csv").exists():
        continue
    try:
        src = hf_hub_download(f"{cfg.HF_USER}/geonusaf-{t}", f"results_{t}.csv", token=HF_TOKEN)
        shutil.copy(src, Path(cfg.OUT_DIR) / f"results_{t}.csv")
    except Exception:
        pass

pub = pd.DataFrame([dict(zip(["method"] + CLASS_NAMES + ["mIoU"], r)) for r in PUBLISHED])
pub["protocol"] = "Test (published, 80k iters)"

ours = []
for f in sorted(Path(cfg.OUT_DIR).glob(f"results_*loveda*{cfg.PROTOCOL}.csv")):
    d = pd.read_csv(f).iloc[0]
    if bool(d.get("incomplete", False)):
        print(f"!! skipping {d['run']} -- schedule incomplete, not a result"); continue
    if cfg.PROTOCOL not in str(d["run"]):
        print(f"!! skipping {d['run']} -- different training protocol, not comparable"); continue
    ours.append({"method": f"OURS: {d['preset']}", **{n: 100 * d[f"IoU_{n}"] for n in CLASS_NAMES},
                 "mIoU": 100 * d["mIoU"],
                 "protocol": f"Val (ours, {int(d['steps'])} steps, 1x T4, seed {cfg.SEED})"})

tbl = pd.concat([pub, pd.DataFrame(ours)], ignore_index=True) if ours else pub
print(tbl.round(2).to_string(index=False))

if len(ours) == 2:
    a = next(o for o in ours if "tcsegformer" in o["method"])
    b = next(o for o in ours if "segformer-b0" in o["method"])
    print(f"\nTHE CLAIM: TC-SegFormer {a['mIoU']:.2f} vs SegFormer-B0 {b['mIoU']:.2f} "
          f"= {a['mIoU']-b['mIoU']:+.2f} mIoU under an identical budget")
    for n in ("Road", "Water", "Barren"):
        print(f"   {n:<8} {a[n]:.2f} vs {b[n]:.2f}  ({a[n]-b[n]:+.2f})")
    print("\nA delta under ~1 point on a single seed is inside the noise. Say so, or run more seeds.")
else:
    print("\nRun the other PRESET to complete the pair -- a single row is not the claim.")

print("\n" + "=" * 96)
print("CAPTION TEXT -- copy this into the paper, do not paraphrase it away:")
print("  Published results are official LoveDA TEST-set scores obtained with substantially longer")
print("  training schedules on datacentre hardware. Our rows are VALIDATION-set scores from a fixed")
print(f"  {cfg.MAX_STEPS}-step single-GPU budget (NVIDIA T4, batch {cfg.BATCH_SIZE}, seed {cfg.SEED}),")
print("  intended as a controlled comparison between our model and its own baseline rather than a")
print("  leaderboard entry. The two protocols are not directly comparable.")
print("=" * 96)
tp = Path(cfg.OUT_DIR) / "loveda_comparison.csv"; tbl.to_csv(tp, index=False); hf_upload(tp, tp.name)

## 23. Standalone inference and official test-set export

Two things in one section.

**Standalone inference** rebuilds the architecture from the config embedded in `best.pt`, so the
checkpoint alone is sufficient — paste it into an empty notebook with nothing above it.

**Test export** runs the best checkpoint over the 1,796 `Test/` tiles and writes single-channel PNGs
with the labels shifted **back** to LoveDA's 1-7 convention (`pred + 1`), flat inside a zip. That is
the format the official evaluation server expects. Getting the `+1` wrong here is the mirror image of
the section 5 bug and produces a submission that scores near zero with no error message.

Set `RUN_TEST_EXPORT = True` when you actually want the ~10 min of inference.

In [ ]:
RUN_TEST_EXPORT = False        # flip to True to build the submission zip

if RUN_TEST_EXPORT and by_split.get("Test"):
    sub_dir = Path(cfg.OUT_DIR) / "test_pred"; sub_dir.mkdir(exist_ok=True, parents=True)
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(by_split["Test"]), cfg.EVAL_BATCH), desc="test export"):
            chunk = by_split["Test"][i:i + cfg.EVAL_BATCH]
            xs = []
            for rec in chunk:
                im = cv2.imread(rec["img"], cv2.IMREAD_COLOR)[:, :, ::-1].astype(np.float32) / 255.0
                xs.append(((im - IMNET_MEAN) / IMNET_STD).transpose(2, 0, 1))
            x = torch.from_numpy(np.ascontiguousarray(np.stack(xs))).to(DEVICE).to(memory_format=torch.channels_last)
            pred = forward_eval(model, x, cfg.SLIDE_STRIDE).argmax(1).cpu().numpy().astype(np.uint8)
            for rec, p in zip(chunk, pred):
                # shift 0..6 BACK to LoveDA's 1..7 convention. Omitting this scores ~0.
                cv2.imwrite(str(sub_dir / f"{rec['stem']}.png"), p + 1)
    zp = Path(cfg.OUT_DIR) / f"submission_{RUN_TAG}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(sub_dir.glob("*.png")):
            z.write(p, arcname=p.name)
    print("submission ->", zp, f"({zp.stat().st_size/1e6:.1f} MB, {len(list(sub_dir.glob('*.png')))} PNGs)")
    chk = cv2.imread(str(next(sub_dir.glob('*.png'))), cv2.IMREAD_UNCHANGED)
    print("sanity: exported values", np.unique(chk), "-- must be within 1..7, never 0")
    hf_upload(zp, zp.name)
else:
    print("test export skipped (RUN_TEST_EXPORT=False or no Test/ split found)")

print(r'''
# ===================== STANDALONE -- paste into an empty notebook =====================
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from huggingface_hub import hf_hub_download
from transformers import SegformerConfig, SegformerForSemanticSegmentation

REPO = "''' + HF_REPO + r'''"
DEV  = "cuda" if torch.cuda.is_available() else "cpu"
NAMES = ["Background","Building","Road","Water","Barren","Forest","Agriculture"]

ck  = torch.load(hf_hub_download(REPO, "best.pt"), map_location="cpu", weights_only=False)
C   = ck["cfg"]

class _Detail(nn.Module):
    def __init__(s, chs):
        super().__init__(); c1, c2, c3 = chs
        b = lambda i, o, st: nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False),
                                           nn.GroupNorm(min(8, o), o), nn.ReLU(True))
        s.net = nn.Sequential(b(3, c1, 2), b(c1, c2, 1), b(c2, c3, 1)); s.out_ch = c3
    def forward(s, x): return s.net(x)

class _TC(nn.Module):
    def __init__(s, C):
        super().__init__()
        conf = SegformerConfig.from_pretrained(C["BACKBONE"]); conf.num_labels = C["NUM_CLASSES"]
        base = SegformerForSemanticSegmentation.from_pretrained(C["BACKBONE"], config=conf,
                                                                ignore_mismatched_sizes=True)
        s.encoder, s.head, s.use_detail = base.segformer, base.decode_head, C["USE_DETAIL"]
        if s.use_detail:
            s.proj = nn.Conv2d(conf.decoder_hidden_size, C["FUSE_CH"], 1)
            s.detail = _Detail(C["DETAIL_CH"]); s.gate = nn.Parameter(torch.tensor(0.1))
            s.fuse = nn.Sequential(nn.Conv2d(C["FUSE_CH"] + s.detail.out_ch, C["FUSE_CH"], 3, 1, 1, bias=False),
                                   nn.GroupNorm(8, C["FUSE_CH"]), nn.ReLU(True))
            s.refine = nn.Conv2d(C["FUSE_CH"], C["NUM_CLASSES"], 1)
    def _fused(s, hs):
        bs = hs[-1].shape[0]; o = []
        for h, mlp in zip(hs, s.head.linear_c):
            x = mlp(h).permute(0, 2, 1).reshape(bs, -1, h.shape[2], h.shape[3])
            o.append(F.interpolate(x, size=hs[0].shape[2:], mode="bilinear", align_corners=False))
        x = s.head.linear_fuse(torch.cat(o[::-1], 1))
        return s.head.dropout(s.head.activation(s.head.batch_norm(x)))
    def forward(s, x):
        H, W = x.shape[-2:]
        f = s._fused(s.encoder(x, output_hidden_states=True, return_dict=True).hidden_states)
        lg = F.interpolate(s.head.classifier(f), (H, W), mode="bilinear", align_corners=False)
        if not s.use_detail: return lg
        g = F.interpolate(s.proj(f), (H // 2, W // 2), mode="bilinear", align_corners=False)
        r = s.refine(s.fuse(torch.cat([g, s.gate * s.detail(x)], 1)))
        return lg + F.interpolate(r, (H, W), mode="bilinear", align_corners=False)

m = _TC(C).to(DEV); m.load_state_dict({k: v.to(DEV) for k, v in ck["model"].items()}, strict=True); m.eval()
MEAN = np.array([0.485,0.456,0.406], np.float32); STD = np.array([0.229,0.224,0.225], np.float32)

@torch.no_grad()
def slide(net, img, crop=512, stride=341, n=7):
    B, _, H, W = img.shape
    hg = max(H-crop+stride-1,0)//stride + 1; wg = max(W-crop+stride-1,0)//stride + 1
    acc = img.new_zeros((B,n,H,W)); cnt = img.new_zeros((B,1,H,W))
    for hi in range(hg):
        for wi in range(wg):
            y2=min(hi*stride+crop,H); x2=min(wi*stride+crop,W); y1=max(y2-crop,0); x1=max(x2-crop,0)
            with torch.cuda.amp.autocast(): lg = net(img[:,:,y1:y2,x1:x2])
            acc[:,:,y1:y2,x1:x2] += lg.float().softmax(1); cnt[:,:,y1:y2,x1:x2] += 1
    return acc/cnt

def predict(path):
    # MUST use the sliding window: the model is trained on 512 crops and whole-tile
    # inference at 1024 costs several mIoU points.
    im = cv2.imread(path, cv2.IMREAD_COLOR)[:, :, ::-1].astype(np.float32) / 255.0
    x = torch.from_numpy(((im - MEAN) / STD).transpose(2, 0, 1))[None].to(DEV)
    with torch.no_grad():
        return slide(m, x).argmax(1)[0].cpu().numpy().astype(np.uint8)    # 0..6; add 1 for LoveDA format
''')

## 24. Qualitative triptychs — best and worst validation tiles

Ranked by per-image mIoU, not sampled at random, and **split by domain** so you get the worst urban
tile and the worst rural tile rather than three of whichever domain happens to be harder.

The failure cases are what reviewers want to see, and on LoveDA they are diagnostic in a specific way:
if the worst tiles are dominated by Barren-vs-Agriculture and Barren-vs-Background confusion, that is
the same spectral ambiguity that caps every published Barren IoU in the teens — a dataset property,
not a bug in your model, and the paper should say so rather than apologising for it.

In [ ]:
import matplotlib.pyplot as plt
# LoveDA visualisation palette (RGB) -- display only, never used for training
PAL_RGB = np.array([[255, 255, 255], [255, 0, 0], [255, 255, 0], [0, 0, 255],
                    [159, 129, 183], [0, 255, 0], [255, 195, 128]], np.uint8)

def colorize(idx):
    out = np.zeros(idx.shape + (3,), np.uint8)
    for c in range(cfg.NUM_CLASSES): out[idx == c] = PAL_RGB[c]
    return out                                     # ignore stays black

picks = []
for dom in sorted({r["domain"] for r in VAL_ROWS}):
    pi = sorted([p for p in per_image if p["domain"] == dom], key=lambda d: d["mIoU"])
    picks += [(f"worst {dom}", r) for r in pi[:2]] + [(f"best {dom}", r) for r in pi[-1:]]

fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.4 * len(picks)))
axes = np.atleast_2d(axes)
for ax_row, (tag, rec) in zip(axes, picks):
    r = VAL_ROWS[rec["i"]]
    img = cv2.imread(r["img"], cv2.IMREAD_COLOR)[:, :, ::-1]
    gt = encode_mask(cv2.imread(r["msk"], cv2.IMREAD_UNCHANGED))
    x = torch.from_numpy(((img.astype(np.float32) / 255.0 - IMNET_MEAN) / IMNET_STD).transpose(2, 0, 1))[None].to(DEVICE)
    with torch.no_grad():
        pr = forward_eval(model, x.to(memory_format=torch.channels_last),
                          cfg.SLIDE_STRIDE).argmax(1)[0].cpu().numpy().astype(np.uint8)
    for ax, im, t in zip(ax_row, [img, colorize(gt), colorize(pr)],
                         [f'{tag} {r["stem"]}\nmIoU {rec["mIoU"]:.3f}', "ground truth", "prediction"]):
        ax.imshow(im); ax.set_title(t, fontsize=8); ax.axis("off")
plt.tight_layout()
qp = Path(cfg.OUT_DIR) / f"qualitative_{RUN_TAG}.png"; plt.savefig(qp, dpi=130, bbox_inches="tight")
hf_upload(qp, qp.name); plt.show()
print("palette order:", dict(zip(CLASS_NAMES, PAL_RGB.tolist())))

## 25. Normalized confusion matrix

Row-normalized, so the diagonal is producer's accuracy.

The three cells to look at first on LoveDA are **Barren -> Agriculture**, **Barren -> Background** and
**Forest -> Agriculture**. Barren is the class every published method scores worst on, and whether
your errors are Barren leaking into Agriculture (under-prediction) or the reverse (over-prediction) is
exactly what the UA/PA columns in section 20 disambiguate. Discuss the magnitude, don't just show the
picture.

In [ ]:
cmn = np.array(M_FULL["cm_norm"])
fig, ax = plt.subplots(figsize=(7.0, 6.0))
im = ax.imshow(cmn, cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(cfg.NUM_CLASSES)); ax.set_yticks(range(cfg.NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
for i in range(cfg.NUM_CLASSES):
    for j in range(cfg.NUM_CLASSES):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cmn[i, j] < 0.55 else "black")
ax.set_title(f"{RUN_TAG} -- row-normalized confusion (LoveDA val)", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
cp2 = Path(cfg.OUT_DIR) / f"confusion_{RUN_TAG}.png"; plt.savefig(cp2, dpi=130, bbox_inches="tight")
hf_upload(cp2, cp2.name); plt.show()
if WB:
    WB.log({"qualitative": wandb.Image(str(qp)), "confusion": wandb.Image(str(cp2))}, step=global_step)
    WB.finish()

## 26. Run checklist

**Before pressing run**

- [ ] LoveDA Kaggle dataset attached; `DATA_ROOT` points at it. Internet ON.
- [ ] `HF_TOKEN` (write) and `WANDB_API_KEY` in Kaggle Secrets.
- [ ] `PRESET` set for this session. **Nothing else touched** — the toggles resolve from it.
- [ ] Section 4 printed 2522 / 1669 / 1796. Anything else means a partial mirror.
- [ ] Section 5 printed the class-share table and **no** assertion fired. This is the `0 -> 255` remap check.
- [ ] Section 7's per-sample timing is under ~40 ms. If not, raise `NUM_WORKERS` — do **not** disable CSA, that changes the model.
- [ ] Section 9 did not print the saturation warning.

**Run both presets. One row is not a result.**

| Run | `PRESET` | What it is |
|---|---|---|
| 1 | `tcsegformer` | the proposed model |
| 2 | `segformer-b0` | the control, identical budget |

**Cost estimate (single T4, batch 16, B0 backbone, 512 crops)**

| Item | Estimate |
|---|---|
| Section 6 mask scan (once, then cached) | 1-2 min |
| 20,000 training steps | ~3.0-4.5 h |
| 20 selection evaluations (400 tiles @ 1024) | ~25 min total |
| Section 20 full evaluation (1,669 tiles @ 1024) | ~5 min |
| Section 23 test export (optional) | ~10 min |
| **One preset, end to end** | **~4-5 h** |
| **Both presets** | **~8-10 h — two sessions, comfortably** |

**Sanity bands for the reported mIoU** (see section 22 for the published rows)

| Value | Reading |
|---|---|
| < 45 | Something is broken. Check section 5 first — it is almost always the label remap. |
| 47-50 | Credible; sits with the classical CNN baselines. Publishable as a controlled comparison. |
| 50-52 | Good for a B0-scale model on this budget. |
| 52+ | Competitive with UNetFormer, which is already one of your five part-1 baselines. |

**Before it goes in the paper**

- [ ] Both runs finished their cosine (`incomplete == False` in the CSV).
- [ ] The section 22 caption text is copied in verbatim — Val vs Test, 20k vs 80k, 1x T4 vs datacentre.
- [ ] Barren IoU is reported and discussed, not buried in the per-class table.
- [ ] The urban/rural gap is reported.
- [ ] The single-seed caveat is stated: a delta under ~1 mIoU point is inside the noise.
- [ ] `core_mIoU` is **not** in the comparison table. It is a Kathmandu-specific diagnostic.